In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
from config.config import ActiveLearningConfig
print(f'ActiveLearningConfig is imported')
from src.active_learning import ActiveLearningSystem

PROJECT_ROOT: /home3/vzcl68/Code/Active_Learning_Benchmarking
ActiveLearningConfig is imported


In [2]:
config = ActiveLearningConfig()

# ---- dataset ----
config.dataset_type = "deepcrack"   # or deepcrack
config.data_dir = "/home3/vzcl68/Datasets/DeepCrack/"
config.img_size = 256
# ---- model ----
config.task = 'segmentation'
config.model_type = "unet"                  # or "maskrcnn"
config.num_classes = 2

# ---- active learning ----
config.cold_start_strategy = "random"
config.query_strategy = "uncertainty"
config.query_size = 10
config.initial_labeled_size = 20

# ---- training ----
config.epochs = 3
config.batch_size = 4
config.num_workers = 2
config.lr = 1e-3
config.weight_decay = 1e-4


# ---- logging ----
config.use_wandb = False

In [3]:
al_system = ActiveLearningSystem(config)

print("Initial labeled set:", len(al_system.labeled_indices))
print("Initial unlabeled pool:", len(al_system.unlabeled_indices))

[DeepCrack] 300 samples loaded for split='train'
[DeepCrack] 237 samples loaded for split='val'


2026-01-29 18:03:25,872 - experiment_1 - INFO - Initialized with 30 labeled and 270 unlabeled samples


Applying cold start strategy: random
Active Learning System initialized with:
  Device: cuda
  Cold Start Strategy: random
  Query Strategy: uncertainty
  Initial labeled: 30 samples
Initial labeled set: 30
Initial unlabeled pool: 270


In [4]:
print("🚀 Initial training")
metrics = al_system.train(epochs=3)

🚀 Initial training

Training cycle 0 with 30 samples
Epoch 1:
  Training time: 20.8s
  AP@[IoU=0.50:0.95]: 0.0000
Epoch 2:
  Training time: 6.3s
  AP@[IoU=0.50:0.95]: 0.0000
Epoch 3:
  Training time: 6.5s
  AP@[IoU=0.50:0.95]: 0.0000


In [5]:
print("🔍 Querying new samples")
new_samples = al_system.query(query_size=10)

print("Newly selected samples:", len(new_samples))
print("Now labeled:", len(al_system.labeled_indices))
print("Now unlabeled:", len(al_system.unlabeled_indices))

🔍 Querying new samples


AttributeError: 'UNetModel' object has no attribute 'eval'

In [ ]:
import matplotlib.pyplot as plt

for idx in new_samples[:3]:
    img, target = al_system.dataset_train[idx]

    plt.figure(figsize=(4,4))
    plt.imshow(img.permute(1,2,0))
    plt.title(f"Queried sample {idx}")
    plt.axis("off")
    plt.show()

In [ ]:
# Initialize system
al_system = ActiveLearningSystem(config)

# Run a single cycle
print("Running initial training...")
metrics = al_system.train(epochs=3)

print("Querying new samples...")
selected_samples = al_system.query()

print(f"Selected {len(selected_samples)} new samples")